In [1]:
import pandas as pd

data_frame = pd.read_csv('output/1.10_preprocessed_train.csv')


In [2]:
import numpy as np

df = data_frame.copy()
df.head()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Age__was_missing,Cabin__was_missing,Embarked__was_missing,...,Cabin_F G63,Cabin_F G73,Cabin_F2,Cabin_F33,Cabin_F38,Cabin_F4,Cabin_G6,Cabin_T,Embarked_Q,Embarked_S
0,1,0,3,22.0,1.0,0,7.2500,0,1,0,...,False,False,False,False,False,False,False,False,False,True
1,2,1,1,38.0,1.0,0,65.6344,0,0,0,...,False,False,False,False,False,False,False,False,False,False
2,3,1,3,26.0,0.0,0,7.9250,0,1,0,...,False,False,False,False,False,False,False,False,False,True
3,4,1,1,35.0,1.0,0,53.1000,0,0,0,...,False,False,False,False,False,False,False,False,False,True
4,5,0,3,35.0,0.0,0,8.0500,0,1,0,...,False,False,False,False,False,False,False,False,False,True


# 1.12 Classification Algorithms (KNN + DT + SVC + Logistic Regression)

Goal: **train and compare** multiple classification algorithms using different hyperparameters, then select the best one based on **accuracy**.

Algorithms:
- KNN (K-Nearest Neighbors)
- Decision Tree
- SVC (Support Vector Classifier)
- Logistic Regression

## Step 1 — Choose the target (y) and features (X)

We try to detect the target column automatically. If your target has a different name, just edit `target_candidates`.

In [3]:
# Detect target column
target_candidates = ['Survived', 'target', 'Target', 'label', 'y']
target = next((c for c in target_candidates if c in df.columns), None)

if target is None:
    raise ValueError('No target column found. Please set target manually.')

target

'Survived'

In [4]:
# Build X and y
X = df.drop(columns=[target]).copy()
y = df[target].copy()

# Keep numeric columns only (your preprocessed file should already be numeric)
X = X.select_dtypes(include=[np.number]).copy()

X.shape, y.shape

((891, 11), (891,))

## Step 2 — Train/Test split
We evaluate the final accuracy on a held-out test set.

In [5]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

X_train.shape, X_test.shape

((712, 11), (179, 11))

## Step 3 — Hyperparameter tuning (GridSearchCV)

We tune each algorithm with a **small grid** (simple and fast for students).

Note:
- KNN, SVC, Logistic Regression often benefit from scaling → we use a Pipeline with `StandardScaler`.
- Decision Tree does not need scaling.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

models = {
    'KNN': (
        Pipeline([('scaler', StandardScaler()), ('model', KNeighborsClassifier())]),
        {
            'model__n_neighbors': [3, 5, 7, 9],
            'model__weights': ['uniform', 'distance'],
        },
    ),
    'DecisionTree': (
        DecisionTreeClassifier(random_state=RANDOM_STATE),
        {
            'max_depth': [None, 3, 5, 10],
            'min_samples_split': [2, 5, 10],
        },
    ),
    'SVC': (
        Pipeline([('scaler', StandardScaler()), ('model', SVC())]),
        {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['linear', 'rbf'],
            'model__gamma': ['scale', 'auto'],
        },
    ),
    'LogisticRegression': (
        Pipeline([('scaler', StandardScaler()), ('model', LogisticRegression(max_iter=2000))]),
        {
            'model__C': [0.1, 1, 10],
            'model__penalty': ['l2'],
            'model__solver': ['lbfgs'],
        },
    ),
}

results = []
best_estimators = {}

for name, (estimator, param_grid) in models.items():
    grid = GridSearchCV(estimator, param_grid=param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    best = grid.best_estimator_
    best_estimators[name] = best

    y_pred = best.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)

    results.append({
        'model': name,
        'best_cv_accuracy': grid.best_score_,
        'test_accuracy': test_acc,
        'best_params': grid.best_params_,
    })

results

c:\Users\abder\anaconda3\envs\mlcourse\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


[{'model': 'KNN',
  'best_cv_accuracy': np.float64(0.7359893627499261),
  'test_accuracy': 0.6480446927374302,
  'best_params': {'model__n_neighbors': 9, 'model__weights': 'distance'}},
 {'model': 'DecisionTree',
  'best_cv_accuracy': np.float64(0.7331823106470994),
  'test_accuracy': 0.664804469273743,
  'best_params': {'max_depth': 3, 'min_samples_split': 2}},
 {'model': 'SVC',
  'best_cv_accuracy': np.float64(0.7402442627794741),
  'test_accuracy': 0.6703910614525139,
  'best_params': {'model__C': 10,
   'model__gamma': 'scale',
   'model__kernel': 'rbf'}},
 {'model': 'LogisticRegression',
  'best_cv_accuracy': np.float64(0.731773859942874),
  'test_accuracy': 0.659217877094972,
  'best_params': {'model__C': 0.1,
   'model__penalty': 'l2',
   'model__solver': 'lbfgs'}}]

## Step 4 — Compare models (Accuracy)
We sort by test accuracy and choose the best one.

In [7]:
import pandas as pd

results_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False)
results_df

,model,best_cv_accuracy,test_accuracy,best_params
2,SVC,0.740244,0.670391,"{'model__C': 10, 'model__gamma': 'scale', 'mod..."
1,DecisionTree,0.733182,0.664804,"{'max_depth': 3, 'min_samples_split': 2}"
3,LogisticRegression,0.731774,0.659218,"{'model__C': 0.1, 'model__penalty': 'l2', 'mod..."
0,KNN,0.735989,0.648045,"{'model__n_neighbors': 9, 'model__weights': 'd..."


In [8]:
best_name = results_df.iloc[0]['model']
best_model = best_estimators[best_name]
best_name

'SVC'

## (Optional) Detailed report for the best model
Confusion matrix + classification report on the test set.

In [9]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_best = best_model.predict(X_test)
print('Best model:', best_name)
print('Test accuracy:', accuracy_score(y_test, y_pred_best))
print('Confusion matrix:\n', confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

Best model: SVC
Test accuracy: 0.6703910614525139
Confusion matrix:
 [[85 25]
 [34 35]]
              precision    recall  f1-score   support

           0       0.71      0.77      0.74       110
           1       0.58      0.51      0.54        69

    accuracy                           0.67       179
   macro avg       0.65      0.64      0.64       179
weighted avg       0.66      0.67      0.67       179

